In [1]:
import json
import numpy as np
import tensorflow as tf

from pathlib import Path


# ==========================================
# 1. DATASET CONFIGURATION
# ==========================================

DATASET_DIR = Path(
    "../datasets/MPII Dataset/filtered_exercise_dataset/person_training_dataset"
)

IMAGE_DIR = DATASET_DIR / "images"
HEATMAP_DIR = DATASET_DIR / "heatmaps"

SPLIT_FILE = DATASET_DIR / "dataset_split.json"
ANNOTATIONS_FILE = DATASET_DIR / "annotations.json"


# ==========================================
# 2. MODEL CONFIGURATION
# ==========================================

IMAGE_SIZE = 256
HEATMAP_SIZE = 64
NUM_JOINTS = 16

BATCH_SIZE = 16
RANDOM_SEED = 42


# ==========================================
# 3. LOAD DATASET INFORMATION
# ==========================================

with open(SPLIT_FILE, "r") as f:
    splits = json.load(f)

with open(ANNOTATIONS_FILE, "r") as f:
    annotations = json.load(f)


# Create person_id -> annotation mapping

annotation_map = {
    item["person_id"]: item
    for item in annotations["annotations"]
}


# ==========================================
# 4. LOAD INDIVIDUAL SAMPLE
# ==========================================

def load_sample(person_id):

    # Retrieve annotation
    annotation = annotation_map[person_id]

    # Retrieve image filename
    image_name = annotation["image_name"]

    image_path = IMAGE_DIR / image_name

    # Check image existence
    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    # Load RGB image
    image = tf.keras.utils.load_img(
        image_path,
        color_mode="rgb",
        target_size=(IMAGE_SIZE, IMAGE_SIZE)
    )

    image = tf.keras.utils.img_to_array(image)

    # Normalize image pixels to [0, 1]
    image = image.astype(np.float32) / 255.0


    # ======================================
    # 5. LOAD HEATMAPS AND VISIBILITY
    # ======================================

    heatmap_name = Path(image_name).stem + ".npz"

    heatmap_path = HEATMAP_DIR / heatmap_name

    if not heatmap_path.exists():
        raise FileNotFoundError(f"Heatmap not found: {heatmap_path}")

    with np.load(heatmap_path) as data:

        heatmaps = data["heatmaps"].astype(np.float32)

        visibility = data["visibility"].astype(np.int8)


    # ======================================
    # 6. VISIBILITY-BASED MASKING
    # ======================================

    # Visible joint  = 1
    # Invisible joint = 0
    # Missing joint = -1

    # Only visible joints contribute to loss
    joint_mask = (visibility == 1).astype(np.float32)


    # ======================================
    # 7. VALIDATE SAMPLE DIMENSIONS
    # ======================================

    expected_heatmap_shape = (
        NUM_JOINTS,
        HEATMAP_SIZE,
        HEATMAP_SIZE
    )

    if heatmaps.shape != expected_heatmap_shape:
        raise ValueError(
            f"Invalid heatmap shape for {person_id}: "
            f"{heatmaps.shape}"
        )

    if visibility.shape != (NUM_JOINTS,):
        raise ValueError(
            f"Invalid visibility shape for {person_id}: "
            f"{visibility.shape}"
        )


    # ======================================
    # 8. TRANSPOSE HEATMAP DIMENSIONS
    # ======================================

    # (16, 64, 64) -> (64, 64, 16)

    heatmaps = np.transpose(heatmaps, (1, 2, 0))


    return image, heatmaps, joint_mask


# ==========================================
# 9. DATA GENERATOR
# ==========================================

def data_generator(person_ids):

    for person_id in person_ids:

        yield load_sample(person_id)


# ==========================================
# 10. CREATE TENSORFLOW DATASET
# ==========================================

def create_dataset(person_ids, shuffle=False):

    dataset = tf.data.Dataset.from_generator(

        lambda: data_generator(person_ids),

        output_signature=(

            tf.TensorSpec(
                shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
                dtype=tf.float32
            ),

            tf.TensorSpec(
                shape=(HEATMAP_SIZE, HEATMAP_SIZE, NUM_JOINTS),
                dtype=tf.float32
            ),

            tf.TensorSpec(
                shape=(NUM_JOINTS,),
                dtype=tf.float32
            )
        )
    )

    # Shuffle training samples
    if shuffle:

        dataset = dataset.shuffle(
            buffer_size=len(person_ids),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True
        )

    # Create batches
    dataset = dataset.batch(BATCH_SIZE)

    # Prefetch batches
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


# ==========================================
# 11. CREATE TRAINING AND VALIDATION DATASETS
# ==========================================

train_dataset = create_dataset(
    splits["train"],
    shuffle=True
)

val_dataset = create_dataset(
    splits["validation"],
    shuffle=False
)


# ==========================================
# 12. DATASET INFORMATION
# ==========================================

print("\nDATASET INITIALIZED SUCCESSFULLY")

print("----------------------------------")

print("Total samples:", len(annotations["annotations"]))

print("Training samples:", len(splits["train"]))

print("Validation samples:", len(splits["validation"]))

print("Batch size:", BATCH_SIZE)

print("Image size:", IMAGE_SIZE)

print("Heatmap size:", HEATMAP_SIZE)

print("Number of joints:", NUM_JOINTS)


# ==========================================
# 13. VERIFY ONE TRAINING BATCH
# ==========================================

for images, heatmaps, joint_masks in train_dataset.take(1):

    print("\nBATCH VERIFICATION")

    print("----------------------------------")

    print("Images shape:", images.shape)

    print("Heatmaps shape:", heatmaps.shape)

    print("Joint masks shape:", joint_masks.shape)

    print(
        "Image value range:",
        tf.reduce_min(images).numpy(),
        tf.reduce_max(images).numpy()
    )

    print(
        "Heatmap value range:",
        tf.reduce_min(heatmaps).numpy(),
        tf.reduce_max(heatmaps).numpy()
    )

    print(
        "Visible joints in batch:",
        tf.reduce_sum(joint_masks).numpy()
    )

    print(
        "Missing/invisible joints in batch:",
        tf.size(joint_masks).numpy()
        - tf.reduce_sum(joint_masks).numpy()
    )

print("\nDATA LOADER READY!")


DATASET INITIALIZED SUCCESSFULLY
----------------------------------
Total samples: 3637
Training samples: 2909
Validation samples: 728
Batch size: 16
Image size: 256
Heatmap size: 64
Number of joints: 16

BATCH VERIFICATION
----------------------------------
Images shape: (16, 256, 256, 3)
Heatmaps shape: (16, 64, 64, 16)
Joint masks shape: (16, 16)
Image value range: 0.0 1.0
Heatmap value range: 0.0 1.0
Visible joints in batch: 182.0
Missing/invisible joints in batch: 74.0

DATA LOADER READY!
